# 01 | Data audit and exploratory analysis

## Project: Marketing decision workspace

This project explores a dashboard for small businesses that brings marketing and sales data together. Its goal is to help a marketer spot measurement problems, understand sales patterns and decide which marketing questions to test next.

This notebook examines the starting dataset before any dashboard or model is built. I will check what each file contains, whether the files connect reliably and which business questions the data can support.

**Data source:** [Marketing and E-commerce Analytics Dataset on Kaggle](https://www.kaggle.com/datasets/geethasagarbonthu/marketing-and-e-commerce-analytics-dataset).

**Important limitation:** This dataset is synthetic. Findings illustrate how the proposed tool could work; they are not evidence of real marketing impact.

## 1. Inspect the source files


In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw")

for name in ["campaigns", "customers", "events", "products", "transactions"]:
    preview = pd.read_csv(data_dir / f"{name}.csv", nrows=5)
    print(f"\n{name}: {list(preview.columns)}")
    display(preview)


campaigns: ['campaign_id', 'channel', 'objective', 'start_date', 'end_date', 'target_segment', 'expected_uplift']


,campaign_id,channel,objective,start_date,end_date,target_segment,expected_uplift
0,1,Paid Search,Cross-sell,2021-10-25,2021-11-26,Deal Seekers,0.022
1,2,Email,Retention,2021-10-24,2021-12-24,Deal Seekers,0.116
2,3,Email,Reactivation,2023-10-08,2023-11-30,Churn Risk,0.100
3,4,Display,Reactivation,2022-07-25,2022-10-07,Deal Seekers,0.111
4,5,Social,Acquisition,2022-07-09,2022-09-29,New Customers,0.144



customers: ['customer_id', 'signup_date', 'country', 'age', 'gender', 'loyalty_tier', 'acquisition_channel']


,customer_id,signup_date,country,age,gender,loyalty_tier,acquisition_channel
0,1,2021-04-08,BR,48,Male,Bronze,Referral
1,2,2023-04-28,IN,36,Female,Silver,Organic
2,3,2022-12-18,UK,35,Female,Silver,Organic
3,4,2022-04-26,US,45,Male,Silver,Paid Search
4,5,2022-04-20,IN,53,Male,Silver,Organic



events: ['event_id', 'timestamp', 'customer_id', 'session_id', 'event_type', 'product_id', 'device_type', 'traffic_source', 'campaign_id', 'page_category', 'session_duration_sec', 'experiment_group']


,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,115.1,Control
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,32.4,Variant_A
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,190.7,Variant_A
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,134.8,Variant_B
4,5,2021-05-10 12:03:09,92735,60646,bounce,NaN,desktop,Email,26,PLP,53.1,Variant_A



products: ['product_id', 'category', 'brand', 'base_price', 'launch_date', 'is_premium']


,product_id,category,brand,base_price,launch_date,is_premium
0,1,Grocery,Brand_58,14.19,2021-08-02,0
1,2,Fashion,Brand_1,25.80,2021-09-14,0
2,3,Electronics,Brand_70,165.46,2021-01-18,1
3,4,Fashion,Brand_56,75.45,2023-03-03,1
4,5,Sports,Brand_1,72.50,2022-04-19,1



transactions: ['transaction_id', 'timestamp', 'customer_id', 'product_id', 'quantity', 'discount_applied', 'gross_revenue', 'campaign_id', 'refund_flag']


,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0


## 2. Check file sizes

Count rows in each CSV to identify the largest tables.

In [2]:
for name in ["campaigns", "customers", "events", "products", "transactions"]:
    path = data_dir / f"{name}.csv"
    with path.open(encoding="utf-8") as file:
        row_count = sum(1 for line in file) - 1  # subtract the header
    print(f"{name}: {row_count:,} rows")

campaigns: 50 rows
customers: 100,000 rows
events: 2,000,000 rows
products: 2,000 rows
transactions: 103,127 rows


## 3. Load the core tables

Load campaigns, customers, products and transactions. Check row counts and data types before assessing data quality. Review the larger events table separately.

In [4]:
campaigns = pd.read_csv(data_dir / "campaigns.csv")
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
transactions = pd.read_csv(data_dir / "transactions.csv")

for name, table in {
    "campaigns": campaigns,
    "customers": customers,
    "products": products,
    "transactions": transactions,
}.items():
    print(f"\n{name}: {table.shape[0]:,} rows, {table.shape[1]} columns")
    display(table.dtypes)


campaigns: 50 rows, 7 columns


campaign_id          int64
channel             object
objective           object
start_date          object
end_date            object
target_segment      object
expected_uplift    float64
dtype: object


customers: 100,000 rows, 7 columns


customer_id             int64
signup_date            object
country                object
age                     int64
gender                 object
loyalty_tier           object
acquisition_channel    object
dtype: object


products: 2,000 rows, 6 columns


product_id       int64
category        object
brand           object
base_price     float64
launch_date     object
is_premium       int64
dtype: object


transactions: 103,127 rows, 9 columns


transaction_id        int64
timestamp            object
customer_id           int64
product_id          float64
quantity              int64
discount_applied    float64
gross_revenue       float64
campaign_id           int64
refund_flag           int64
dtype: object

 ## 4. Check missing values and duplicates

Check incomplete fields, repeated rows and duplicate IDs in the core tables.

In [5]:
tables = {
    "campaigns": campaigns,
    "customers": customers,
    "products": products,
    "transactions": transactions,
}

id_columns = {
    "campaigns": "campaign_id",
    "customers": "customer_id",
    "products": "product_id",
    "transactions": "transaction_id",
}

for name, table in tables.items():
    missing = table.isna().sum()
    print(f"\n{name}")
    print(f"missing values: {missing.sum():,}")
    print(f"duplicate rows: {table.duplicated().sum():,}")
    print(f"duplicate {id_columns[name]}: {table[id_columns[name]].duplicated().sum():,}")
    display(missing[missing > 0])


campaigns
missing values: 0
duplicate rows: 0
duplicate campaign_id: 0


Series([], dtype: int64)


customers
missing values: 0
duplicate rows: 0
duplicate customer_id: 0


Series([], dtype: int64)


products
missing values: 0
duplicate rows: 0
duplicate product_id: 0


Series([], dtype: int64)


transactions
missing values: 20,898
duplicate rows: 0
duplicate transaction_id: 0


product_id       10449
gross_revenue    10449
dtype: int64

## 5. Investigate incomplete transactions

Check whether missing product and revenue values occur in the same records and whether they relate to refunds.

In [6]:
missing_product = transactions["product_id"].isna()
missing_revenue = transactions["gross_revenue"].isna()

print("missing both:", (missing_product & missing_revenue).sum())
print("missing product only:", (missing_product & ~missing_revenue).sum())
print("missing revenue only:", (~missing_product & missing_revenue).sum())

display(pd.crosstab(
    transactions["refund_flag"],
    missing_revenue,
    rownames=["refund_flag"],
    colnames=["gross_revenue missing"]
))

missing both: 10449
missing product only: 0
missing revenue only: 0


gross_revenue missing,False,True
refund_flag,,
0,89974,10124
1,2704,325


### Finding

Product and revenue are missing together in 10,449 transactions (10.1%). Most are not refunds. Revenue analyses must report how many transactions were excluded.

In [7]:
display(transactions.loc[missing_revenue].head(10))

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
15,16,2021-02-13 07:36:17,33154,NaN,1,0.05,NaN,29,0
20,21,2023-03-29 19:49:38,52652,NaN,1,0.00,NaN,40,0
21,22,2021-08-01 15:47:25,30425,NaN,1,0.15,NaN,49,0
55,56,2023-06-13 04:11:11,1280,NaN,3,0.15,NaN,0,0
57,58,2022-07-23 20:38:49,32032,NaN,1,0.05,NaN,13,0
70,71,2023-06-16 14:48:36,98281,NaN,1,0.00,NaN,38,0
96,97,2023-11-04 18:12:19,89129,NaN,1,0.00,NaN,21,1
98,99,2021-09-13 11:18:26,2817,NaN,4,0.05,NaN,45,0
99,100,2023-02-24 11:20:54,83870,NaN,1,0.00,NaN,18,0
109,110,2022-05-17 05:37:41,3708,NaN,2,0.10,NaN,0,0


## 6. Check date ranges

Convert date fields and compare the periods covered by customers, campaigns and transactions.

In [8]:
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"], errors="coerce")
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
products["launch_date"] = pd.to_datetime(products["launch_date"], errors="coerce")
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="coerce")

date_columns = {
    "campaign start": campaigns["start_date"],
    "campaign end": campaigns["end_date"],
    "customer signup": customers["signup_date"],
    "product launch": products["launch_date"],
    "transaction": transactions["timestamp"],
}

for name, dates in date_columns.items():
    print(f"{name}: {dates.min()} to {dates.max()} | invalid dates: {dates.isna().sum():,}")

campaign start: 2021-01-20 00:00:00 to 2023-11-04 00:00:00 | invalid dates: 0
campaign end: 2021-02-21 00:00:00 to 2024-01-06 00:00:00 | invalid dates: 0
customer signup: 2021-01-01 00:00:00 to 2023-12-31 00:00:00 | invalid dates: 0
product launch: 2021-01-01 00:00:00 to 2023-12-31 00:00:00 | invalid dates: 0
transaction: 2021-01-01 00:12:50 to 2023-12-31 22:37:32 | invalid dates: 0


## 7. Validate campaign links

Check whether campaign IDs match and whether linked transactions occurred within the campaign dates.

In [9]:
linked = transactions.loc[transactions["campaign_id"].ne(0)].merge(
    campaigns[["campaign_id", "start_date", "end_date"]],
    on="campaign_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

matched = linked.loc[linked["_merge"].eq("both")].copy()
transaction_date = matched["timestamp"].dt.normalize()

outside = (
    transaction_date.lt(matched["start_date"])
    | transaction_date.gt(matched["end_date"])
)

print(f"transactions with a nonzero campaign ID: {len(linked):,}")
print(f"campaign IDs with no match: {linked['_merge'].eq('left_only').sum():,}")
print(f"linked transactions outside campaign dates: {outside.sum():,}")
print(f"share outside campaign dates: {outside.mean():.1%}")

transactions with a nonzero campaign ID: 82,172
campaign IDs with no match: 0
linked transactions outside campaign dates: 78,102
share outside campaign dates: 95.0%


### finding

All nonzero campaign IDs match a campaign, but 78,102 of 82,172 linked transactions (95.0%) fall outside that campaign's dates. Campaign-linked sales are therefore unsuitable for impact or uplift analysis.

## Conclusion

This synthetic dataset is unsuitable for the proposed campaign analysis. Although campaign IDs match, 95.0% of linked transactions occur outside their campaign dates. Campaign-linked sales cannot be interpreted reliably. No impact or uplift analysis will be built from these links.